In [ ]:
import io
import numpy as np
import pandas as pd

# 2. Parsing CSV dengan penanganan desimal koma
df = pd.read_csv(("dataset-susu.csv"), decimal=",")

# 3. Hitung waktu berjalan (elapsed time) per perlakuan sampel (dalam menit)
# Dihitung dari titik awal kemunculan sampel pertama masing-masing kategori
df["t_elapsed_min"] = df.groupby("Sample ID")["Timestamp (ms)"].transform(
    lambda x: (x - x.min()) / 60000.0
)


# 4. Fungsi penentuan kapasitas umur simpan awal (Baseline Lifespan) berdasarkan suhu
def hitung_baseline_shelf_life(suhu):
  if suhu <= 10.0:
    # Interpolasi pendingin: 4°C - 10°fC (2880 menit ~ 48 jam)
    return 2880 - (suhu - 4.0) * 80
  elif suhu <= 28.0:
    # Ruang Standar: 240 - 300 menit (4 - 5 jam)
    return 300 - ((suhu - 20.0) / 8.0) * 60
  elif suhu <= 30.0:
    # Zona Transisi Tropis: 150 - 240 menit
    return 240 - ((suhu - 28.0) / 2.0) * 90
  else:
    # Ruang Tropis / Hangat: 60 - 120 menit (1 - 2 jam)
    decay = 120 - ((suhu - 30.0) / 4.0) * 60
    return max(40, decay)


# 5. Hitung sisa masa simpan (label_shelf_life_min)
def estimasi_sisa_waktu(row):
  suhu = row["Suhu (°C)"]
  t_elapsed = row["t_elapsed_min"]
  ec25 = row["EC25 (mS/cm)"]

  baseline = hitung_baseline_shelf_life(suhu)

  # Penalti ionik: jika EC25 mulai merambat naik > 4.2 mS/cm, kurangi umur simpan
  penalti_ec = 1.0
  if ec25 > 4.2:
    penalti_ec = max(0.85, 1.0 - (ec25 - 4.2) * 0.1)

  sisa_waktu = (baseline * penalti_ec) - t_elapsed
  return max(0, int(round(sisa_waktu)))


df["label_shelf_life_min"] = df.apply(estimasi_sisa_waktu, axis=1)

# 6. Label uji alkohol (seluruh observasi 8 jam negatif)
df["alcohol_test"] = "NEGATIF"


# 7. Penentuan Grade Mutu Berdasarkan Urgensi Sisa Waktu
def tentukan_grade(sisa_menit):
  if sisa_menit > 90:
    return "GRADE_A"  # Mutu prima, waktu simpan masih panjang
  elif sisa_menit >= 30:
    return "GRADE_B"  # Early warning, harus segera didistribusikan/dipasteurisasi
  else:
    return "GRADE_C"  # Kritis / hampir pecah


df["label_grade"] = df["label_shelf_life_min"].apply(tentukan_grade)

# 8. Tampilkan dan simpan hasil
kolom_tampil = [
    "Sample ID",
    "Log ID",
    "Suhu (°C)",
    "EC25 (mS/cm)",
    "alcohol_test",
    "label_grade",
    "label_shelf_life_min",
]
print(df[kolom_tampil].to_string(index=False))

df.to_csv("dataset_susu_labeled.csv", index=False)

Sample ID  Log ID  Suhu (°C)  EC25 (mS/cm) alcohol_test label_grade  label_shelf_life_min
 S_DINGIN      31       8.03         3.550      NEGATIF     GRADE_A                  2558
 S_DINGIN      32       8.03         3.554      NEGATIF     GRADE_A                  2558
 S_DINGIN      33       8.03         3.554      NEGATIF     GRADE_A                  2558
 S_DINGIN      34       8.03         3.556      NEGATIF     GRADE_A                  2558
 S_DINGIN      35       8.06         3.552      NEGATIF     GRADE_A                  2555
  S_RUANG      36      28.73         4.588      NEGATIF     GRADE_A                   199
  S_RUANG      37      28.66         4.585      NEGATIF     GRADE_A                   202
  S_RUANG      38      28.73         4.588      NEGATIF     GRADE_A                   199
  S_RUANG      39      28.69         4.592      NEGATIF     GRADE_A                   201
  S_RUANG      40      28.69         4.597      NEGATIF     GRADE_A                   201
 S_HANGAT 

In [22]:
df_labeled = pd.read_csv('dataset_susu_labeled.csv')
df_labeled = df.drop(['alcohol_test'], axis=1, inplace=True)

df.to_csv("dataset_susu_labeled.csv", index=False)